# Task 2: Final Test Prediction

This notebook applies only the durable best-model package selected in Notebook 05 to the unseen test set. It preserves the prediction-template row order, exports one season label per image, and saves the corresponding score matrix for reproducibility. Model comparison belongs to Notebook 06; this notebook is deliberately restricted to final inference.

## How to Run

Run Notebooks 01–05 before this notebook. Notebook 05 must have exported exactly one selected-model package: `models/task2/task2_model.pt` for a neural winner or `task2_model.joblib` for a Random Forest winner. Notebook 06 is an independent evaluation and is not a computational dependency for test prediction. Then run Sections 1–3 in order. Rerun this notebook whenever the selected package, test images, or prediction template changes.

## Required Structure to Run

The notebook requires the durable selected-model package, prediction template, and test images. The template must contain an `id` column, every ID must have a matching JPEG image, and exactly one selected-model package should exist.

```text
Machine-Learning-Assignment-2/
├── pyproject.toml
├── src/
│   ├── preprocessing.py
│   └── task2_utils.py
├── models/task2/
│   └── task2_model.pt  OR  task2_model.joblib
└── datasets/test/
    ├── styles_prediction.csv
    └── images_test/
        └── <id>.jpg
```

The notebook creates `outputs/task2/task2_predictions.csv` and `outputs/task2/task2_test_scores.npy`. No external comparison model or labelled validation data is required for final prediction.

## 1. Prediction Setup

In [1]:
%matplotlib inline
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from IPython.display import display

REPO_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / 'pyproject.toml').exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError('Could not find the repository root containing pyproject.toml')
sys.path.insert(0, str(REPO_ROOT))

from src.preprocessing import TEST_IMAGE_DIR, load_image_array
from src.task2_utils import (
    TASK2_MODEL_DIR, TASK2_OUTPUT_DIR, NeuralTrainer,
    build_task2_neural_model, ensure_task2_directories,
    extract_visual_features,
)

ensure_task2_directories()
print('Repository root:', REPO_ROOT)

Repository root: /Users/s4030327/Machine-Learning-Assignment-2


## 2. Load the Notebook 05 Selected-Model Package

Notebook 05 packages the winning estimator together with its class order, image geometry, normalisation statistics, model configuration, validation evidence, and preprocessing fingerprint. Loading this package instead of reconstructing the winner from individual experiment files prevents prediction from silently using a different architecture or preprocessing contract.

In [2]:
neural_package_path = TASK2_MODEL_DIR / 'task2_model.pt'
forest_package_path = TASK2_MODEL_DIR / 'task2_model.joblib'
available_packages = [
    path for path in (neural_package_path, forest_package_path) if path.exists()
]
if not available_packages:
    raise FileNotFoundError('Missing selected-model package; run Notebook 05 first')
if len(available_packages) > 1:
    raise RuntimeError(
        'Both neural and Random Forest selected-model packages exist. '
        'Remove the stale package or rerun Notebook 05 so the winner is unambiguous.'
    )

selected_model_path = available_packages[0]
if selected_model_path.suffix == '.pt':
    package = NeuralTrainer._load(selected_model_path)
    model_kind = 'neural'
else:
    package = joblib.load(selected_model_path)
    model_kind = 'random_forest'

required_keys = {
    'selected_model', 'fingerprint', 'model_config', 'classes',
    'image_target_size', 'normalisation_mean', 'normalisation_std',
}
missing_keys = sorted(required_keys.difference(package))
if missing_keys:
    raise ValueError(f'Selected-model package is missing metadata: {missing_keys}')

CLASSES = list(package['classes'])
TARGET = package.get('target', 'season')
IMAGE_TARGET_SIZE = tuple(package['image_target_size'])
NORM_MEAN = np.asarray(package['normalisation_mean'], dtype=np.float32)
NORM_STD = np.asarray(package['normalisation_std'], dtype=np.float32)
if len(CLASSES) < 2 or NORM_MEAN.shape != (3,) or NORM_STD.shape != (3,):
    raise ValueError('Selected-model package contains an invalid class or normalisation contract')
if not np.isfinite(NORM_MEAN).all() or not np.isfinite(NORM_STD).all() or np.any(NORM_STD <= 0):
    raise ValueError('Selected-model package contains invalid normalisation statistics')

print('Selected model:', package['selected_model'])
print('Package:', selected_model_path)
print('Run fingerprint:', package['fingerprint'])
print('Class order:', CLASSES)

Selected model: DenseNet-121
Package: /Users/s4030327/Machine-Learning-Assignment-2/models/task2/task2_model.pt
Run fingerprint: 5f507c7bb57f
Class order: ['Fall', 'Spring', 'Summer', 'Winter']


## 3. Selected-model test prediction

This section runs the model selected in Notebook 05 on every test image. Images receive the same deterministic resize and padding used for training. A neural winner is additionally scaled and normalised with the training-only statistics stored in its package; a Random Forest winner receives the same engineered feature extractor used during training. The prediction CSV preserves the template row order, while the NumPy file preserves the corresponding probabilities or neural logits for reproducibility and later auditing.

In [3]:
template_path = REPO_ROOT / 'datasets' / 'test' / 'styles_prediction.csv'
template = pd.read_csv(template_path)
if 'id' not in template.columns:
    raise ValueError(f'{template_path} must contain an id column')
test_paths = [Path(TEST_IMAGE_DIR) / f'{image_id}.jpg' for image_id in template['id']]
missing = [path for path in test_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f'{len(missing)} test images are missing')

if model_kind == 'random_forest':
    model = package['model']
    features = np.vstack([
        extract_visual_features(
            load_image_array(path, IMAGE_TARGET_SIZE, scale=False)
        )
        for path in test_paths
    ])
    test_scores = model.predict_proba(features)
    predicted = np.asarray(model.classes_, dtype=int)[test_scores.argmax(axis=1)]
else:
    model = build_task2_neural_model(package['model_config'], len(CLASSES))
    model.load_state_dict(package['state_dict'])
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
    model = model.to(device).eval()
    mean = torch.tensor(NORM_MEAN, device=device).view(1, 3, 1, 1)
    std = torch.tensor(NORM_STD, device=device).view(1, 3, 1, 1)
    chunks = []
    with torch.no_grad():
        for start in range(0, len(test_paths), 256):
            arrays = np.stack([
                load_image_array(path, IMAGE_TARGET_SIZE, scale=False)
                for path in test_paths[start:start + 256]
            ])
            images = torch.from_numpy(arrays).to(device).permute(0, 3, 1, 2).float() / 255
            chunks.append(model((images - mean) / std).cpu())
    test_scores = torch.cat(chunks).numpy()
    predicted = test_scores.argmax(axis=1)

if test_scores.shape != (len(template), len(CLASSES)) or not np.isfinite(test_scores).all():
    raise ValueError(f'Invalid test-score matrix: {test_scores.shape}')
predictions = template.copy()
predictions[TARGET] = [CLASSES[int(index)] for index in predicted]
prediction_path = TASK2_OUTPUT_DIR / 'task2_predictions.csv'
score_path = TASK2_OUTPUT_DIR / 'task2_test_scores.npy'
predictions.to_csv(prediction_path, index=False)
np.save(score_path, test_scores)
print('Inference device:', device if model_kind == 'neural' else 'CPU / scikit-learn')
print('Saved predictions:', prediction_path)
print('Saved model scores:', score_path)
display(predictions.head())

FileNotFoundError: [Errno 2] No such file or directory: '/Users/s4030327/Machine-Learning-Assignment-2/datasets/test/styles_prediction.csv'

**Prediction finding.** In the recorded DenseNet-121 run, the selected model produced predictions for all 5,829 test images. Summer accounts for 3,274 predictions (56.2%), broadly reflecting its dominance in training. Spring accounts for 1,106 predictions (19.0%) despite representing only 4.1% of the training labels. Because the test labels are unavailable, this distribution is a diagnostic observation rather than evidence of error or accuracy.